In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Create directory
!mkdir -p /content/bobiac_data_spotiflow
# Download the data
!wget https://github.com/bobiac/bobiac-book/releases/download/data-bobiac-2026/04_05_06_07_seg_and_spot.zip -O /content/bobiac_data_spotiflow/04_05_06_07_seg_and_spot.zip
# Unzip the data, remove zip file and macOS metadata files (if any)
!cd /content/bobiac_data_spotiflow && unzip 04_05_06_07_seg_and_spot.zip && rm -f 04_05_06_07_seg_and_spot.zip && rm -rf __MACOSX

In [ ]:
!pip install cellpose
!pip install matplotlib
!pip install tqdm
!pip install tifffile
!pip install spotiflow

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile
from spotiflow.model import Spotiflow
from tqdm import tqdm

In [ ]:
def save_points_as_csv(points, output_path="points.csv", channel_last=False) -> None:
    """Save points as a napari-compatible CSV (drag-and-drop as Points layer).

    napari maps the CSV columns (axis-0, axis-1, ...) to the layer axes in order.
    `predict_multichannel` returns the channel as the *last* column (e.g. (y, x, channel)),
    so set `channel_last=True` to move it to the front (e.g. (channel, y, x)) and have the
    spots line up with a channel-first (C, ...) image in napari.
    """
    points = np.asarray(points)
    if channel_last:
        # move the last column (channel) to the front
        points = points[:, [-1, *range(points.shape[1] - 1)]]
    ndim = points.shape[1]
    headers = ["index"] + [f"axis-{i}" for i in range(ndim)]
    with open(output_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        for i, p in enumerate(points):
            writer.writerow([i, *p])

def plot_points_on_image(image, points, is_3d=False, skip_channels=None):
    """Overlay detected spots on each image channel.

    Works with both predict() (single-channel, no channel column) and
    predict_multichannel() (channel index as last column) outputs.

    Parameters
    ----------
    image : array-like
        2D: (Y, X) or (C, Y, X).  3D: (Z, Y, X) or (C, Z, Y, X).
    points : array-like
        From predict():              (y, x) for 2D,  (z, y, x) for 3D.
        From predict_multichannel(): (y, x, ch) for 2D, (z, y, x, ch) for 3D.
    is_3d : bool
        If True, max-projects along Z before plotting.
    skip_channels : list of int, optional
        Channel indices on which points should NOT be drawn.
    """
    image = np.asarray(image)
    points = np.asarray(points)
    skip_channels = set(skip_channels or [])

    # Detect multichannel by whether points have an extra channel column
    spatial_cols = 3 if is_3d else 2
    is_multichannel = points.shape[1] > spatial_cols

    if is_3d:
        if image.ndim == 3:  # (Z, Y, X) — single channel
            proj = image.max(axis=0)[np.newaxis]  # -> (1, Y, X)
        else:  # (C, Z, Y, X) — multichannel
            proj = image.max(axis=1)  # -> (C, Y, X)
        y_col, x_col = 1, 2
    else:
        if image.ndim == 2:  # (Y, X) — single channel
            proj = image[np.newaxis]  # -> (1, Y, X)
        else:  # (C, Y, X) — multichannel
            proj = image
        y_col, x_col = 0, 1

    n_channels = proj.shape[0]
    ch_col = points.shape[1] - 1  # only meaningful when is_multichannel

    n_cols = min(n_channels, 3)
    n_rows = (n_channels + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    axes = np.array(axes).flatten()

    colors = ["green", "magenta", "cyan", "yellow", "orange", "red"]
    for ch in range(n_channels):
        ax = axes[ch]
        ax.imshow(proj[ch], cmap="gray")
        if ch not in skip_channels:
            ch_pts = points[points[:, ch_col] == ch] if is_multichannel else points
            ax.scatter(
                ch_pts[:, x_col],
                ch_pts[:, y_col],
                s=10,
                facecolors=colors[ch % len(colors)],
                # facecolors="none",
                # edgecolors=colors[ch % len(colors)],
                linewidths=0.5,
                marker="x",
            )
            ax.set_title(f"Channel {ch}  ({len(ch_pts)} spots)")
        else:
            ax.set_title(f"Channel {ch}  (skipped)")
        ax.axis("off")

    # Hide unused axes in the last row
    for ch in range(n_channels, len(axes)):
        axes[ch].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
image_path = "content/bobiac_data_spotiflow/05_spot_detection_spotiflow/2d_spots.tif"
image = tifffile.imread(image_path)

In [ ]:
print(image.shape)

In [ ]:
ch = 6
plt.figure(figsize=(10, 5))
for i in range(ch):
    plt.subplot(2, ch // 2, i + 1)
    plt.imshow(image[i], cmap="gray")
    plt.title(f"Channel {i + 1}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
model = Spotiflow.from_pretrained("general")

In [ ]:
tr_image = image.transpose(1, 2, 0)  # (C, Y, X) -> (Y, X, C)
print(tr_image.shape)

In [ ]:
points, details = model.predict_multichannel(tr_image, channels=(2, 3, 4))

In [ ]:
print(points.shape)

In [ ]:
print(points[0])

In [ ]:
plot_points_on_image(image, points, skip_channels=[0, 1])

In [ ]:
save_points_as_csv(points, "2d_points.csv", channel_last=True)

In [ ]:
# `ch` is channel index, change it to visualize the heatmap of a different channel.
# note that this is the channel index as passed to predict_multichannel().
ch = 0

plt.imshow(details[ch].heatmap, cmap="hot")
plt.colorbar(label="Probability")
plt.title(f"Heatmap for Channel {ch}")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
plt.hist(details[0].intens, bins=15, color="green", alpha=0.5, label="channel 3")
plt.hist(details[1].intens, bins=15, color="magenta", alpha=0.5, label="channel 4")
plt.hist(details[2].intens, bins=15, color="cyan", alpha=0.5, label="channel 5")
plt.hist(details[3].intens, bins=15, color="yellow", alpha=0.5, label="channel 6")
plt.xlabel("Intensity")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
# Path to the folder containing the images
folder_path = Path("content/bobiac_data_spotiflow/05_spot_detection_spotiflow")  # change this to your folder path

# Create a subfolder to save the spot detection results
spots_folder = folder_path / "spots_coords"
spots_folder.mkdir(exist_ok=True)

# Get the sorted list of all .tif images in the folder
images_path = sorted(folder_path.glob("*.tif"))

# Initialize the model once before the loop
model = Spotiflow.from_pretrained("general")

# specify the channels you want to process in predict_multichannel()
channels = (2, 3, 4)

# NOTE: tqdm is used to show a progress bar, but you can remove it if you don't want it
for image_path in tqdm(images_path, desc="Processing images"):
    # Load the image
    image = tifffile.imread(image_path)
    # Transpose the image to channel-last format for `predict_multichannel`
    tr_image = image.transpose(1, 2, 0)  # (C, Y, X) -> (Y, X, C)
    # Run Spotiflow on the image
    points, details = model.predict_multichannel(tr_image, channels=channels)
    # Save the points as a CSV file
    output_path = spots_folder / f"{image_path.stem}_points.csv"
    save_points_as_csv(points, str(output_path), channel_last=True)

In [ ]:
# Create directory
!mkdir -p /content/bobiac_data_spotiflow_3d
# Download the data
!wget https://github.com/bobiac/bobiac-book/releases/download/data-bobiac-2026/05_spot_detection_spotiflow_3d.zip -O /content/bobiac_data_spotiflow/05_spot_detection_spotiflow_3d.zip
# Unzip the data, remove zip file and macOS metadata files (if any)
!cd /content/bobiac_data_spotiflow && unzip 05_spot_detection_spotiflow_3d.zip && rm -f 05_spot_detection_spotiflow_3d.zip && rm -rf __MACOSX

In [ ]:
image_path = "content/bobiac_data_spotiflow_3d/05_spot_detection_spotiflow_3d/3d_spots.tif"
image_3d = tifffile.imread(image_path)

print(image_3d.shape)

In [ ]:
plt.imshow(image_3d.max(axis=0), cmap="gray")
plt.title("Max projection (Z)")
plt.axis("off")
plt.show()

In [ ]:
model = Spotiflow.from_pretrained("smfish_3d")

In [ ]:
points, details = model.predict(image_3d)

In [ ]:
points[0]  # z, y, x

In [ ]:
plot_points_on_image(image_3d, points, is_3d=True)

In [ ]:
save_points_as_csv(points, "3d_points.csv")

In [ ]:
plt.imshow(details.heatmap.max(axis=0), cmap="magma")
plt.title("Heatmap (max projection)")
plt.colorbar(label="Probability")
plt.axis("off")
plt.show()

In [ ]:
plt.hist(details.intens, bins=15)
plt.xlabel("Intensity")
plt.ylabel("Frequency")
plt.show()